# `WGA151CI` - All Initial Gubbins Events Processing

# Import statements

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns
#import pickle
from tqdm import tqdm

%matplotlib inline

In [2]:
# https://bioframe.readthedocs.io/en/latest/guide-intervalops.html
import bioframe as bf


In [3]:
import json

import ete3 as ETE

from ete3 import Tree


# Import custom utils functions

In [4]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
#from gcutils.general import parse_PAFtools_VarTSV, label_DF_ByOvrLapGenes, label_PAF_DF_ByOvrLapGenes'

from gcutils.gubbinsfuncs import get_RecombEvents_From_Gubbins_GFF

from gcutils.gubbinsfuncs import parse_BaseReconstruction_EMBL_Gubbins, annotate_Gubbins_SNP_Events_By_RecombEventID, annotate_Gubbins_SNP_Events_By_EventID_And_ParalogRegion

from gcutils.gubbinsfuncs import get_RecombEvents_From_Gubbins_GFF_V2, label_Gubbins_Events_DF_ByOvrLap_H37RvGenes, label_Gubbins_Events_DF_ByOvrLap_HHRs

from gcutils.gubbinsfuncs import annotate_HHR_with_GCE_overlaps


from gcutils.gubbinsfuncs import Parse_Process_Gubbins_Events_Standard, add_HighHomologyAndRepeat_OvrlapInfo_ToGubbinsEvents


from gcutils.general import check_overlap_with_gene_group 

from gcutils.gubbinsfuncs import Add_AA_Consequences_ToGubbinsSNPs_DF

from gcutils.gubbinsfuncs import get_BranchLengths_and_DescendantCounts_FromTree

from gcutils.gubbinsfuncs import annotate_tree_with_lineages, infer_InternalNode_Lineages

from gcutils.gubbinsfuncs import subset_EventsDF_ForGenes


### Import functions - parsimony (`fitch`) scoring of all mutations from GC Events

In [5]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

from gcutils.parsimony import get_parsimony_stats_AllVariants_Per_GC_Event

# from gcutils.parsimony import summarize_parsimony_across_sites, parsimony_per_snp

# from gcutils.parsimony import fitch_parsimony_score, parsimony_from_df_for_site #parsimony_score_from_dataframe

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### Set matplotlib text export settings for Adobe Illustrator

In [6]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

#### Pandas Viewing Settings

In [7]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

# Import/parse processed H37rv genome annotations

In [8]:
RepoRef_Dir = "../../References"

AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir = f"{RepoRef_Dir}/201027_H37rv_AnnotatedGenes_And_IntergenicRegions"
#H37Rv_GenomeAnnotations_Genes_TSV = f"{AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir}/H37Rv_GenomeAnnotations.Genes.tsv"
H37Rv_GenomeAnnotations_Genes_V2_TSV = f"{AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir}/H37Rv_GenomeAnnotations.Genes.V2.tsv"

## H37Rv Gene Annotations TSV
H37Rv_GenomeAnno_Genes_DF = pd.read_csv(H37Rv_GenomeAnnotations_Genes_V2_TSV, sep = "\t")
#H37Rv_GenomeAnno_Genes_DF["Middle"] = (H37Rv_GenomeAnno_Genes_DF["Start"] + H37Rv_GenomeAnno_Genes_DF["End"]) / 2
#H37Rv_GenomeAnno_Genes_DF["Length"]  = H37Rv_GenomeAnno_Genes_DF["End"] - H37Rv_GenomeAnno_Genes_DF["Start"]

H37Rv_GeneInfo_Subset_DF = H37Rv_GenomeAnno_Genes_DF[["H37rv_GeneID", "Symbol", "Feature", "Functional_Category", "Is_Pseudogene", "Product", "PEandPPE_Subfamily", "ExcludedGroup_Category"]]

RvID_To_Symbol_Dict = dict(H37Rv_GeneInfo_Subset_DF[['H37rv_GeneID', 'Symbol']].values)
Symbol_To_RvID_Dict = dict(H37Rv_GenomeAnno_Genes_DF[['Symbol', 'H37rv_GeneID']].values)
Symbol_To_FuncCat_Dict = dict(H37Rv_GenomeAnno_Genes_DF[['Symbol', 'Functional_Category']].values)

ESX_Genes_List_TSV = f"{RepoRef_Dir}/190927_H37rv_ListOf_ESXgenes.tsv"
Esx_Genes_DF = pd.read_csv(ESX_Genes_List_TSV, sep = '\t')

In [9]:
H37Rv_GenomeAnno_Genes_DF.head(1)

,Chrom,Start,End,Strand,H37rv_GeneID,Symbol,Feature,Functional_Category,Is_Pseudogene,Product,PEandPPE_Subfamily,ExcludedGroup_Category,Gene_Cat_V2
0,NC_000962.3,0,1524,+,Rv0001,dnaA,CDS,information pathways,No,Chromosomal replication initiator protein DnaA,NaN,NotExcluded,information pathways


## Define relevant H37Rv gene lists for analysis (PE/PPE, Esx, 13E12 gene)

In [10]:
ListOf_Esx_Symbols = list(Esx_Genes_DF["symbol"].values)
ListOf_Esx_RvIDs = list(Esx_Genes_DF["gene_id"].values)

In [11]:
listOf_PEPPE_Symbols = list( H37Rv_GenomeAnno_Genes_DF.query(" Functional_Category == 'PE/PPE' ")["Symbol"].values )
listOf_PEPPE_RvIDs = list( H37Rv_GenomeAnno_Genes_DF.query(" Functional_Category == 'PE/PPE' ")["H37rv_GeneID"].values )

In [12]:
# listOf_13E12_Region_RvIDs = ["Rv0094c", "Rv0095c", "Rv0393", "Rv1572c", "Rv1572c", "Rv1128c", "Rv1148c", "Rv1587c", "Rv1588c", "Rv1702c", "Rv1945", "Rv2100", "Rv3466", "Rv3467"]  
listOf_REP13E12_Region_RvIDs = ["Rv0094c", "Rv0095c", "Rv0393", "Rv1572c", "Rv1572c", "Rv1128c", "Rv1148c", "Rv1587c", "Rv1588c", "Rv1702c", "Rv1945", "Rv2100", "Rv3466", "Rv3467"]  

# Parse `LowComplexity` and `Low-Mappability` Regions of H37Rv

In [13]:
RepoRef_Dir = "../../References"
Rv_longdust_LCB_OutDir = f"{RepoRef_Dir}/H37Rv_Longdust_LowComplexityRegions"
Rv_LowPupMap_OutDir = f"{RepoRef_Dir}/H37Rv_MappabilityAnalysis_K50E4"

Rv_LowPupmap_K50E4_TSV = f"{Rv_LowPupMap_OutDir}/H37Rv.LowPileupMap.Below1.K50_E4.tsv"
Rv_LowComplexityRegions_TSV = f"{Rv_longdust_LCB_OutDir}/H37Rv.longdust.LCR.relaxedshorter.tsv"


In [14]:
Rv_LowPmap_DF = pd.read_csv(Rv_LowPupmap_K50E4_TSV, sep ="\t")
Rv_LowPmap_DF.shape

(1092, 5)

In [15]:
Rv_LCRs_DF = pd.read_csv(Rv_LowComplexityRegions_TSV, sep ="\t")
Rv_LCRs_DF.shape

(231, 5)

In [16]:
Rv_LowPmap_DF["Length"].sum()

189147

In [17]:
Rv_LCRs_DF["Length"].sum()

127160

# Parse H37Rv Reference sequences (Genome, Genes, Proteins)

## Parse H37Rv genome sequence (DNA)

In [18]:
from Bio import SeqIO


In [19]:
H37rv_Ref_GBK_PATH = "/n/data1/hms/dbmi/farhat/mm774/References/GCF_000195955.2_ASM19595v2_genomic.gbk"
H37Rv_FA = "/n/data1/hms/dbmi/farhat/mm774/References/GCF_000195955.2_ASM19595v2_genomic.fasta"

H37Rv_Seq = SeqIO.read(H37Rv_FA, "fasta").seq
len(H37Rv_Seq)

4411532

## Parse H37Rv Protein (AA) and gene (DNA) sequences

In [20]:
O2_RefDir = "/n/data1/hms/dbmi/farhat/mm774/References"

MycoBrowser_RefFiles_Dir = f"{O2_RefDir}/190619_Mycobrowser_H37rv_ReferenceFiles"

H37Rv_Proteins_MycoBro_FAA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.fasta"
H37Rv_Proteins_MycoBro_FAA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.esxM_Added.fasta"
H37Rv_Proteins_NCBI_FAA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.fasta"
H37RV_Genes_FA = f"{MycoBrowser_RefFiles_Dir}/Mycobacterium_tuberculosis_H37Rv_genes_v3.fasta"



H37Rv_FAA_PATH = f"{O2_RefDir}/GCF_000195955.2_ASM19595v2_proteins.faa"
H37Rv_FAA_PATH = f"{O2_RefDir}/GCF_000195955.2_ASM19595v2_proteins.esxM_Added.faa"
H37Rv_GBK_PATH = f"{O2_RefDir}/GCF_000195955.2_ASM19595v2_genomic.gbk"


In [21]:
!ls -1 $MycoBrowser_RefFiles_Dir

Mycobacterium_tuberculosis_H37Rv_genes_v3.fasta
Mycobacterium_tuberculosis_H37Rv_genome_v3.fasta
Mycobacterium_tuberculosis_H37Rv_genome_v3.fasta.fai
Mycobacterium_tuberculosis_H37Rv_gff_v3.gff
Mycobacterium_tuberculosis_H37Rv_gff_v3.REP13E12_Regions.gff
Mycobacterium_tuberculosis_H37Rv_proteins_v3.fasta
Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.esxM_Added.fasta
Mycobacterium_tuberculosis_H37Rv_proteins_v3_TrimmedHeader.fasta
Mycobacterium_tuberculosis_H37Rv_txt_v3_PEPPE_subfamilies.txt
Mycobacterium_tuberculosis_H37Rv_txt_v3.txt.tsv


### Parse MycoBrowser Protein Seq Ref

In [22]:
dictOf_H37Rv_MycoBrow_ProtSeq = {}

for index, record in tqdm(enumerate(SeqIO.parse(H37Rv_Proteins_MycoBro_FAA, "fasta"))):
    ShortID = record.name
    
    dictOf_H37Rv_MycoBrow_ProtSeq[ShortID] = record.seq


4091it [00:00, 176595.46it/s]


### Parse MycoBrowser Gene Seq Ref

In [23]:
dictOf_H37Rv_MycoBrow_Gene_Seq = {}

for index, record in tqdm(enumerate(SeqIO.parse(H37RV_Genes_FA, "fasta"))):

    ShortID = record.name.split("|")[0]
    dictOf_H37Rv_MycoBrow_Gene_Seq[ShortID] = record.seq


4187it [00:00, 121325.00it/s]


In [24]:
list(dictOf_H37Rv_MycoBrow_Gene_Seq.keys())[:2]

['Rv3728', 'Rv3729']

### Parse NCBI Protein Seq Ref

In [25]:
dictOf_H37Rv_ProtSeq = {}
dictOf_H37Rv_ProtRecord = {}

for index, record in tqdm(enumerate(SeqIO.parse(H37Rv_FAA_PATH, "fasta"))):
    Rec_Description = record.description
    dict_Attr = {}
    for i in Rec_Description.split(" "):
        ###Just looking for line with " " character (as key = value)
        if "=" in i:
            key = i.strip().split("=")[0].strip('"').strip('[')
            value = i.strip().split("=")[1].strip('"').strip(']')
            ###Put them in a dictionnary
            dict_Attr[key]=value
    
    ShortID = dict_Attr["locus_tag"]
    dictOf_H37Rv_ProtSeq[ShortID] = record.seq
    dictOf_H37Rv_ProtRecord[ShortID] = record


3907it [00:00, 87302.19it/s]


In [26]:
#dictOf_H37Rv_ProtSeq["Rv1196"]

In [27]:
#dictOf_H37Rv_ProtSeq["Rv1196"] == dictOf_H37Rv_MycoBrow_ProtSeq["Rv1196"]

In [28]:
dictOf_H37Rv_ProtSeq["Rv1792"]

Seq('MASRFMTDPHAMRDMAGRFEVHAQTVEDEARRMWASAQNISGAGWSGMAEATSLDTMT')

In [29]:
dictOf_H37Rv_MycoBrow_ProtSeq["Rv1792"]

Seq('MASRFMTDPHAMRDMAGRFEVHAQTVEDEARRMWASAQNISGAGWSGMAEATSLDTMT')

# Parse in H37Rv Homology-Map Results (k19w19)

### Define all HmMap file paths

In [30]:
Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V9"

H37_Rv_MM2_HomologyMapping_Dir = f"{Main_Project_Dir}/250901.H37Rv.HomologyMapping.k19w19.ProcessedData.V2"

# Define paths to output TSVS

### Homologous regions (MERGED)
RvHmMap_Merged_ParaRegions_TSV  = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.MergedRegions.ParalogousRegions.k19w19.tsv"
RvHmMap_Merged_LocalRepeats_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.MergedRegions.LocalRepeats.k19w19.tsv"

### Homology map (pairwise alignments)
RvHmMap_Aln_All_TSV           = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.All.tsv"
RvHmMap_Aln_PRs_NoOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.NoOverlap.tsv"
RvHmMap_Aln_LRs_WiOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.WiOverlap.tsv"

RvHmMap_Aln_PRs_NoOverlap_Clustered_TSV     = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.NoOverlap.Clustered.tsv"
RvHmMap_Aln_LRs_WiOverlap_Clustered_TSV     = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.OnlyOverlap.Clustered.tsv"

### Variants from the homology map alignments
RvHmMap_Var_TSV      = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.variants.tsv"
RvHmMap_Var_SNPs_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.variants.snps.tsv"



### Variants from the homology map alignments (For each pairwise alignment between PRs)
RvHmMap_Var_PerAln_TSV      = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.variants.ParalogousRegions.PerAln.tsv"
RvHmMap_Var_PerAln_SNPs_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.variants.ParalogousRegions.PerAln.snps.tsv"



### Parse in HmRegions (`Paralogous_Regions` and `Local_Repeats`)

In [31]:
HmMapRegs_ParaRegs_k19w19_DF = pd.read_csv(RvHmMap_Merged_ParaRegions_TSV,
                                           sep="\t")

HmMapRegs_ParaRegs_k19w19_DF.shape

(200, 15)

In [32]:
HmMapRegs_LocalRepeats_k19w19_DF = pd.read_csv(RvHmMap_Merged_LocalRepeats_TSV, 
                                               sep="\t")
HmMapRegs_LocalRepeats_k19w19_DF.shape

(50, 13)

In [33]:
HmMapRegs_All_LRsPRs_K19w19_DF = pd.concat([HmMapRegs_ParaRegs_k19w19_DF,
                                            HmMapRegs_LocalRepeats_k19w19_DF])

HmMapRegs_All_LRsPRs_K19w19_DF.shape

(250, 15)

#### Peak at head of each HmMap Regions DFs

In [34]:
HmMapRegs_ParaRegs_k19w19_DF.head()

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,num_HmAln_All,num_HmAln_PR_BelowSeqID100,num_HmAln_PR_MinSeqID99,HmRegionNum,HmRegionID,NOvrlap_IS6110_Label,PR_SetID
0,0,NC_000962.3,80184,80523,80353.5,339,Rv0071,0,1,1,1,0,PR_HmRegion_000,0,PR_Set_1
1,1,NC_000962.3,80623,82664,81643.5,2041,"Rv0072,Rv0073",0,1,1,1,1,PR_HmRegion_001,0,PR_Set_2
2,2,NC_000962.3,103705,105130,104417.5,1425,"Rv0094c,Rv0095c",0,2,2,2,2,PR_HmRegion_002,0,PR_Set_3
3,3,NC_000962.3,149571,149808,149689.5,237,PE_PGRS2,0,1,1,1,3,PR_HmRegion_003,0,PR_Set_4
4,4,NC_000962.3,177203,177447,177325.0,244,_,0,1,1,1,4,PR_HmRegion_004,0,PR_Set_5


In [35]:
HmMapRegs_LocalRepeats_k19w19_DF.head()

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,num_HmAln_All,num_HmAln_PR_BelowSeqID100,num_HmAln_PR_MinSeqID99,HmRegionNum,HmRegionID
0,0,NC_000962.3,333811,335879,334845.0,2068,PE_PGRS3,0,1,1,1,0,LR_HmRegion_000
1,1,NC_000962.3,366430,375121,370775.5,8691,"PPE5,PPE6",0,6,6,6,1,LR_HmRegion_001
2,2,NC_000962.3,424011,432951,428481.0,8940,"hspR,PPE7,PPE8",0,5,4,4,2,LR_HmRegion_002
3,3,NC_000962.3,566288,580814,573551.0,14526,"hbhA,Rv0476,Rv0477,deoC,Rv0479c,Rv0480c,Rv0481...",0,4,2,1,3,LR_HmRegion_003
4,4,NC_000962.3,631298,631436,631367.0,138,Rv0538,0,2,2,2,4,LR_HmRegion_004


### Parse in homology-map DFs (pairwise alignments between all homologous regions)

In [36]:
HmMap_Aln_k19w19_DF = pd.read_csv(RvHmMap_Aln_All_TSV,
                           sep="\t")
HmMap_Aln_k19w19_DF.shape

(776, 25)

In [37]:
HmMap_Aln_k19w19_NoOverlap_DF = pd.read_csv(RvHmMap_Aln_PRs_NoOverlap_TSV,
                                     sep="\t")
HmMap_Aln_k19w19_NoOverlap_DF.shape

(640, 38)

In [38]:
HmMap_Aln_k19w19_LocalRepeat_DF = pd.read_csv(RvHmMap_Aln_LRs_WiOverlap_TSV,
                                              sep="\t")
HmMap_Aln_k19w19_LocalRepeat_DF.shape

(136, 35)

### Parse in HomologyMap Alignment Variants DFs

In [39]:
### paralog variants - Improved per alignment `cs` tag approach 
Mtb_HM_Var_PR_DF = pd.read_csv(RvHmMap_Var_PerAln_TSV, sep="\t")
print(Mtb_HM_Var_PR_DF.shape)

Mtb_HM_Var_PR_SNPs_DF = pd.read_csv(RvHmMap_Var_PerAln_SNPs_TSV, sep="\t")
print(Mtb_HM_Var_PR_SNPs_DF.shape)

(60402, 20)
(50203, 20)


In [40]:
# Mtb_HM_Var_DF = pd.read_csv(RvHmMap_Var_TSV, sep="\t")
# Mtb_HM_Var_SNPs_DF = pd.read_csv(RvHmMap_Var_SNPs_TSV, sep="\t")

# print(Mtb_HM_Var_DF.shape)
# print(Mtb_HM_Var_SNPs_DF.shape)

In [41]:
TarCol = ['Target_Name', 'Target_Start', 'Target_End', 'Ref', 'Alt', 'SNP'] 

Mtb_HM_Var_PR_SNPs_Trim_DF = Mtb_HM_Var_PR_SNPs_DF[TarCol]

print(Mtb_HM_Var_PR_SNPs_Trim_DF.shape)

Mtb_HM_Var_PR_SNPs_Trim_TrimUnq_DF = Mtb_HM_Var_PR_SNPs_Trim_DF.drop_duplicates()

print(Mtb_HM_Var_PR_SNPs_Trim_TrimUnq_DF.shape)


(50203, 6)
(41454, 6)


In [42]:
Mtb_HM_Var_PR_SNPs_Trim_TrimUnq_DF.head(3)

,Target_Name,Target_Start,Target_End,Ref,Alt,SNP
0,NC_000962.3,3082657,3082658,T,C,True
1,NC_000962.3,3082660,3082661,C,A,True
2,NC_000962.3,3082667,3082668,G,A,True


# Parse `Mtb151CI` Isolate Metadata

In [43]:
Repo_DataDir = "../../Data"
InputAsmPath_Dir = f"{Repo_DataDir}/231121.InputAsmTSVs.MtbSetV3.151CI"

MtbSetV3_151CI_InputAsmPATHs_TSV = f"{InputAsmPath_Dir}/231121.MtbSetV3.151CI.HybridAndSRAsm.FAPATHs.V1.tsv"
MtbSetV3_151CI_AsmSumm_TSV = f"{InputAsmPath_Dir}/231121.MtbSetV3.151CI.HybridAsm.AsmSummary.V2.tsv"


### Reading in "WGA151CI_AsmSummary_DF"

In [44]:
WGA151CI_AsmSummary_DF = pd.read_csv(MtbSetV3_151CI_AsmSumm_TSV, sep = "\t")

SampleIDs_151CI_SOI = list( WGA151CI_AsmSummary_DF["SampleID"].values )
WGA151CI_SampleIDs = SampleIDs_151CI_SOI
WGA151CI_AsmSummary_DF.shape


(151, 7)

#### Create SampleID Mapping Dicts

In [45]:
WGA151CI_ID_To_PrimLineage_Dict = dict(WGA151CI_AsmSummary_DF[['SampleID', 'PrimaryLineage']].values)
WGA151CI_ID_To_SubLineage_Dict = dict( WGA151CI_AsmSummary_DF[["SampleID", "Lineage"]].values)
WGA151CI_ID_To_Dataset_Dict = dict(WGA151CI_AsmSummary_DF[['SampleID', 'Dataset_Tag']].values)  


In [46]:
SampleID_To_PrimLineage_Dict = WGA151CI_ID_To_PrimLineage_Dict
SampleID_To_SubLineage_Dict = WGA151CI_ID_To_SubLineage_Dict


# Parse `Nuc-Div` results for `WGA-151CI` dataset

## Define NucDiv analysis file paths

In [47]:
AnalysisName = "250201.WGA151CI.V10"

Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V10"

V8_Target_Output_Dir = f"{Main_Project_Dir}/{AnalysisName}"

VCFT_NucDiv_Dir = f"{V8_Target_Output_Dir}/VCFtools_SNV_NucDiv_AcrossH37Rv"

Pickle_PATH_WGA_NucDiv_PI_Dict = VCFT_NucDiv_Dir + f"/{AnalysisName}.SNVs.NucDiv.DictOfResults.pickle"   

#NucDiv_NoFilt_1kb_TSV_PATH = f"{VCFT_NucDiv_Dir}/WGA.SNVs.NucDiv.1000bp.ALL.tsv"
NucDiv_NoFilt_1kb_TSV_PATH = f"{VCFT_NucDiv_Dir}/WGA.SNVs.NucDiv.1000bp.ALL.Anno.tsv"
NucDiv_NoFilt_1kb_V2_TSV_PATH = f"{VCFT_NucDiv_Dir}/WGA.SNVs.NucDiv.1000bp.ALL.Anno.V2.tsv"


### Parse pickle of NucDiv stats

In [48]:
import pickle

In [49]:
with open(Pickle_PATH_WGA_NucDiv_PI_Dict, "rb") as f: NucDiv_PI_Dict = pickle.load(f)
NucDiv_PI_Dict.keys()

dict_keys([('ALL', 1000)])

In [50]:
xx_1kb_All_np = NucDiv_PI_Dict[("ALL", 1000)]["Fill_np"]["XX"]
yy_1kb_All_np = NucDiv_PI_Dict[("ALL", 1000)]["Fill_np"]["YY"] * 1000


### Parse 1-kb NucDiv DF

In [51]:
# Read in TSV
WGA_NucDiv_PI_1kb_DF = pd.read_csv(NucDiv_NoFilt_1kb_V2_TSV_PATH, sep = "\t")

NucDiv_1kb_DF = WGA_NucDiv_PI_1kb_DF

# Calculate mean, median, standard deviation, and Median Absolute Deviation
median_1kb_PI = NucDiv_1kb_DF['NucDiv'].median()
mean_1kb_PI = NucDiv_1kb_DF['NucDiv'].mean()
std_1kb_PI = NucDiv_1kb_DF['NucDiv'].std()
MAD_1kb_PI = np.median(np.abs(NucDiv_1kb_DF['NucDiv'] - median_1kb_PI))


### Define NucDiv Hospots (N = 37)

In [52]:
NucDiv_HSR_1kb_DF = NucDiv_1kb_DF.query("IsNucDivHotspot == True")
NucDiv_HSR_1kb_DF.shape

(37, 20)

# Parse all individual variants per genome annotated by CDS effect (`WGA-151CI`)

In [53]:
Mtb151_AllVar_Dir = "../../Data/241030.Mtb151CI.AllVariants.Anno.V1/"

Mtb151_AllVar_TSVGZ              = f"{Mtb151_AllVar_Dir}/Mtb.151CI.AllVariants.V1.tsv.gz"
Mtb151_AllVar_WiCDSAnno_TSVGZ    = f"{Mtb151_AllVar_Dir}/Mtb.151CI.AllVariants.WiCDSAnno.V1.tsv.gz"
Mtb151_SNPs_WiInterSNPDist_TSVGZ = f"{Mtb151_AllVar_Dir}/Mtb.151CI.AllSNPs.WiInterSNPDists.V1.tsv"


In [54]:

Mtb151CI_AllAsm_AllVar_DF = pd.read_csv(Mtb151_AllVar_WiCDSAnno_TSVGZ, sep="\t")

AllAsm_AllVar_DF = Mtb151CI_AllAsm_AllVar_DF
AllAsm_AllVar_DF.shape

(260869, 20)

In [55]:
AllAsm_AllVar_DF["SampleID"].nunique()

151

In [56]:
AllAsm_AllVar_DF["Variant_Group"].value_counts()

Variant_Group
Missensse_SNP        121836
Synonymous_SNP        96849
Inframe_INDEL         28878
Frameshift_INDEL      11561
PrematureStop_SNP      1745
Name: count, dtype: int64

In [57]:
AllAsm_AllVar_DF['Alt_WithStopCodon'].value_counts()

Alt_WithStopCodon
_        136955
False    122169
True       1745
Name: count, dtype: int64

In [58]:
AllAsm_AllVar_DF['INDEL_Type'].value_counts()

INDEL_Type
INS    15286
DEL    11316
Name: count, dtype: int64

In [59]:
AllAsm_AllVar_DF['frameshift'].value_counts()

frameshift
False    249308
True      11561
Name: count, dtype: int64

# Copy Gubbins output from SMK output dir to analysis dir

### Define directories to assembly + analysis pipeline(s)

In [60]:
### Define directories to PMP-SM (PacBio assembly and analysis pipeline)

### Define PATH to pipeline output directories

Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects"

Mtb_WGA_SMK_Outputs_Dir = Project_Dir + "/Mtb-WGA-SMK-Output"

WGA151CI_SMK_OutputDir = Mtb_WGA_SMK_Outputs_Dir + "/231121_MtbSetV3_151CI"


In [61]:
!ls -alh $WGA151CI_SMK_OutputDir | tail -n 14

drwxrwsr-x 2 mm774 hpc_farhat 4.0K Jan  1  2024 Asm_MergeSNPs_mpileup
drwxrwsr-x 2 mm774 hpc_farhat 4.0K Dec  2  2023 Asm_MergeVar_mpileup
drwxrwsr-x 2 mm774 hpc_farhat 4.0K Nov 22  2023 Busco_Download_Tmp
drwxrwsr-x 2 mm774 hpc_farhat 4.0K Nov 22  2023 FastANI
drwxrwsr-x 2 mm774 hpc_farhat 4.0K Sep  1 10:22 HomologyMapping
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Nov 25  2024 .ipynb_checkpoints
drwxrwsr-x 2 mm774 hpc_farhat 4.0K Mar 16  2024 Minigraph
drwxrwsr-x 2 mm774 hpc_farhat 4.0K Nov 30  2023 NucDiversity
drwxrwsr-x 2 mm774 hpc_farhat 4.0K Nov 22  2023 O2logs
drwxrwsr-x 2 mm774 hpc_farhat 4.0K Dec 10  2024 PanGenome_Analysis
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Dec 27  2024 Pangraph_Mtb151CIWiRv_Analysis
drwxrwsr-x 2 mm774 hpc_farhat 4.0K Jan  1  2024 Phylogenies
drwxrwsr-x 2 mm774 hpc_farhat 4.0K Nov 30  2023 RecombDetection
drwxrwsr-x 2 mm774 hpc_farhat 4.0K Nov 22  2023 SourMash


### Define output dirs of pipeline

In [62]:
target_OutputDir = WGA151CI_SMK_OutputDir

i_Phylogeny_Dir = f"{target_OutputDir}/Phylogenies"

i_NucDiv_Dir = f"{target_OutputDir}/NucDiversity"
i_NucDiv_mpileup_Dir = f"{i_NucDiv_Dir}/NucDiv_SNVs_mpileup"

i_Recomb_Dir = f"{target_OutputDir}/RecombDetection"
i_Gubbins_Out_Dir = f"{i_Recomb_Dir}/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh"


# 1) Define relevant PATHS and target directories

In [63]:
AnalysisName = "250201.WGA151CI.V10"

Gubbins_OutPrefix = "Gubbins"

Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V10"

!mkdir $Main_Project_Dir

Target_Output_Dir = f"{Main_Project_Dir}/{AnalysisName}"

!mkdir $Target_Output_Dir

input_SampleNames = SampleIDs_151CI_SOI


mkdir: cannot create directory ‘/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V10’: File exists
mkdir: cannot create directory ‘/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V10/250201.WGA151CI.V10’: File exists


## Copy Gubbins output directory (From SMK pipeline output) to analysis directory

In [64]:
!cp -r $i_Gubbins_Out_Dir/ $Target_Output_Dir/

In [65]:
#!ls -alh $i_Gubbins_Out_Dir

In [66]:
Gubbins_V1_OutputDir = f"{Target_Output_Dir}/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh"

In [67]:
#!ls -alh $Gubbins_V1_OutputDir

## Define paths to GUBBINS outputs

In [68]:
Gubbins_V1_NEW_ResultsDir = f"{Target_Output_Dir}/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh"

In [69]:
Gubbins_NodeLabelledTree_PATH      = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.node_labelled.final_tree.tre"

Gubbins_BranchStats_CSV_PATH       = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.per_branch_statistics.csv"


Gubbins_RecombPreds_GFF            = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.recombination_predictions.gff"
Gubbins_RecombPreds_EMBL           = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.recombination_predictions.embl"

Gubbins_RecombPreds_RenamedChr_GFF = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.recombination_predictions.RenamedCHR.gff"
Gubbins_RecombPreds_RenamedChr_BED = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.recombination_predictions.RenamedCHR.bed"      

Gubbins_BaseReconstruction_EMBL    = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.branch_base_reconstruction.embl"


In [70]:
print(Gubbins_V1_NEW_ResultsDir)

/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V10/250201.WGA151CI.V10/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh


In [71]:
!ls -alh $Gubbins_V1_NEW_ResultsDir

total 32M
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Dec  9 11:19 .
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Dec  9 10:36 ..
-rw-r--r-- 1 mm774 hpc_farhat  19K Dec  7 18:49 GCE.FitchParsimonyScores.PerEvent.tsv
-rw-r--r-- 1 mm774 hpc_farhat 139K Dec  7 18:49 GCE.FitchParsimonyScores.PerSite.tsv
-rw-r--r-- 1 mm774 hpc_farhat 3.4M Dec  9 11:19 Gubbins.branch_base_reconstruction.AnnoByEvent.All.tsv
-rw-r--r-- 1 mm774 hpc_farhat 387K Dec  9 11:19 Gubbins.branch_base_reconstruction.AnnoByEvent.EventSNPsOnly.tsv
-rw-r--r-- 1 mm774 hpc_farhat 6.9M Dec 12 11:38 Gubbins.branch_base_reconstruction.embl
-rw-r--r-- 1 mm774 hpc_farhat 3.3M Dec  9 11:19 Gubbins.branch_base_reconstruction.V2.AnnoByParalogMatch.All.tsv
-rw-r--r-- 1 mm774 hpc_farhat 3.2M Dec 12 11:38 Gubbins.filtered_polymorphic_sites.fasta
-rw-r--r-- 1 mm774 hpc_farhat 3.2M Dec 12 11:38 Gubbins.filtered_polymorphic_sites.phylip
-rw-r--r-- 1 mm774 hpc_farhat 4.4K Dec 12 11:38 Gubbins.final_tree.tre
-rw-r--r-- 1 mm774 hpc_farhat 263K Dec  9 11:14 Gub

# Begin Processing of Gubbins Results

# Gubbins Phylogeny Processing

In [72]:
## 1) Parse Gubbins Tree w/ ETE3

i_Gubbins_T = Tree(Gubbins_NodeLabelledTree_PATH, format = 1)

## 2) Parsing over each node of the tree (ETE3), get ML BRANCH LENGTH and Number of descendents per node
Tree_BranchLen_Dict, Tree_NumDescendants_PerNode_Dict = get_BranchLengths_and_DescendantCounts_FromTree(i_Gubbins_T)

## 3) Label each LEAF within phylogeny by MTBC Lineage + Infer lineage of each internal node of the tree
i_Gubbins_T = annotate_tree_with_lineages(i_Gubbins_T,
                                          SampleID_To_PrimLineage_Dict)

i_Gubbins_T, Tree_PrimaryLin_PerNode_Dict = infer_InternalNode_Lineages(i_Gubbins_T,
                                                                        SampleID_To_PrimLineage_Dict)

                          



Number of leaves processed: 151


In [73]:
print( len(list(Tree_BranchLen_Dict.keys())) )

300


In [74]:
print( len(list(Tree_NumDescendants_PerNode_Dict.keys())) )

300


In [75]:
print( len(list(Tree_PrimaryLin_PerNode_Dict.keys())) )

295


### Save a JSON map of NodeID to Mtb lineage

#### Output "Tree_PrimaryLin_PerNode_Dict" dictionary 

In [76]:
Gubbins_NodeToPriLineage_Dict_JSON = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.NodeToPrimaryLineage.json"

with open(Gubbins_NodeToPriLineage_Dict_JSON, 'w') as json_file:
    json.dump(Tree_PrimaryLin_PerNode_Dict, json_file)

#### test reading back in the JSON
# with open(Gubbins_NodeToPriLineage_Dict_JSON) as json_file:
#     Tree_PrimaryLin_PerNode_Dict = json.load(json_file)


In [77]:
len(list(Tree_PrimaryLin_PerNode_Dict.keys()))

295

# Parse & Annotate Gubbins Recombination GFFs (Individual Events)

### This gives us info at the individual event level (inferred by Gubbins)

In [78]:
H37Rv_DictOf_HighHomologyAndRepeat_InfoDFs = {"HmMap_Paralogous_Regions": HmMapRegs_ParaRegs_k19w19_DF, 
                                              "HmMap_LocalRepeat_Regions": HmMapRegs_LocalRepeats_k19w19_DF, 
                                              "HmMap_Paralogous_Aln_DF": HmMap_Aln_k19w19_NoOverlap_DF,
                                              "HmMap_LocalRepeat_Aln_DF": HmMap_Aln_k19w19_LocalRepeat_DF, 
                                              "LowComplexity_Regions_DF": Rv_LCRs_DF, 
                                              "LowPmap_Regions_DF": Rv_LowPmap_DF, }


H37Rv_Esx_PEPPE_REP13E12_GeneLists_Dict = {"Esx"      : ListOf_Esx_RvIDs,
                                           "PEPPE"    : listOf_PEPPE_RvIDs,
                                           "REP13E12" : listOf_REP13E12_Region_RvIDs}



In [79]:
Gubbins_Recomb_Events_DF = Parse_Process_Gubbins_Events_Standard(Gubbins_RecombPreds_RenamedChr_GFF,
                                                          Tree_PrimaryLin_PerNode_Dict,
                                                          Tree_NumDescendants_PerNode_Dict,
                                                          H37Rv_GenomeAnno_Genes_DF,
                                                          H37Rv_Esx_PEPPE_REP13E12_GeneLists_Dict,
                                                          H37Rv_DictOf_HighHomologyAndRepeat_InfoDFs)

print( Gubbins_Recomb_Events_DF.shape)


(324, 30)


In [80]:
Gubbins_Recomb_Events_DF.query("Num_Tips_Downstream == 0").shape

(185, 30)

## Output updated Gubbins_Recomb_Events_DF to TSV

In [81]:
Gubbins_RecombPreds_Anno_TSV = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.recombination_predictions.Anno.tsv"

Gubbins_Recomb_Events_DF.to_csv(Gubbins_RecombPreds_Anno_TSV, sep="\t", index=False)


### Explore a bit the detected GC events in the dataset

In [82]:
Gubbins_Recomb_Events_DF["Overlap_Genes"].value_counts().shape

(86,)

#### How many GC events DO occur in a PR HHR?

In [83]:
Gubbins_Recomb_Events_DF.query("N_HmMapAln_PR_Ovrlap > 0").shape

(295, 30)

#### How many GC events DO occur in a ANY HHR?

In [84]:
Gubbins_Recomb_Events_DF.query("N_HmMapAln_LR_Ovrlap > 0").shape

(46, 30)

In [85]:
Gubbins_Recomb_Events_DF.query("N_HmMapAln_PR_Ovrlap > 0 | N_HmMapAln_LR_Ovrlap > 0").shape

(309, 30)

# Count recomb events per region

### Count inferred recombination events (Gubbins) per: <br> 
a) 1 kb region <br> 
b) annotated gene <br> 
c) merged-homologous region <br>

## Read in annotated windows of the H37Rv genome

In [86]:
RepoRef_Dir = "../../References"

H37Rv_Windows_Dir = f"{RepoRef_Dir}/H37Rv_GenomeWindows"
H37Rv_1kb_Win_Anno_TSV = f"{H37Rv_Windows_Dir}/H37Rv.1000bp.Windows.Anno.tsv"

Rv_1kb_Win_DF = pd.read_csv(H37Rv_1kb_Win_Anno_TSV, sep = "\t")
Rv_1kb_Window_Start_To_Genes_Dict = dict(Rv_1kb_Win_DF[['Start', 'Overlap_Genes']].values)


## a) Let's count events per 1 kb window

In [87]:
RvWin_CoordCols = ("Chrom", "Start", "End")
RE_CoordCols = ("seqname", "start_0based", "end_1based")

Rv_1kb_RE_Count_DF = bf.count_overlaps(Rv_1kb_Win_DF,
                                       Gubbins_Recomb_Events_DF,
                                       cols1 = RvWin_CoordCols,
                                       cols2 = RE_CoordCols).rename(columns={'count': 'pGCE_Count'})

Rv_1kb_RE_Count_DF["CenterOfRegion"] = ((Rv_1kb_RE_Count_DF["Start"] + Rv_1kb_RE_Count_DF["End"]) / 2)

Rv_1kb_RE_Count_DF.shape

(4412, 7)

In [88]:
Rv_1kb_RE_Count_DF.sort_values("pGCE_Count", ascending=False).head(4)

,Chrom,Start,End,Overlap_Genes,Middle,pGCE_Count,CenterOfRegion
104,NC_000962.3,104000,105000,"Rv0094c,Rv0095c",104500.0,28,104500.0
3883,NC_000962.3,3883000,3884000,"rmlC,Rv3466,Rv3467",3883500.0,16,3883500.0
1634,NC_000962.3,1634000,1635000,PE_PGRS27,1634500.0,16,1634500.0
103,NC_000962.3,103000,104000,"Rv0093c,Rv0094c",103500.0,15,103500.0


## b) Count events per gene

In [89]:
RE_CoordCols = ("seqname", "start_0based", "end_1based")
GenomeAnno_CoordCols = ("Chrom", "Start", "End")

Rv_Genes_RE_Count_DF = bf.count_overlaps(H37Rv_GenomeAnno_Genes_DF,
                                         Gubbins_Recomb_Events_DF,
                                         cols1 = GenomeAnno_CoordCols,
                                         cols2 = RE_CoordCols).rename(columns={'count': 'pGCE_Count'})

Rv_Genes_RE_Count_DF["CenterOfRegion"] = ((Rv_Genes_RE_Count_DF["Start"] + Rv_Genes_RE_Count_DF["End"]) / 2)

Rv_Genes_RE_Count_DF.shape

(4079, 15)

## c) Count events per PR (MERGED Paralogous region)

In [90]:
RE_CoordCols = ("seqname", "start_0based", "end_1based")
HmRegion_CoordCols = ("Chr", "Start", "End")

Rv_PRs_GCE_Count_DF = bf.count_overlaps(HmMapRegs_ParaRegs_k19w19_DF,
                                        Gubbins_Recomb_Events_DF,
                                        cols1 = HmRegion_CoordCols,
                                        cols2 = RE_CoordCols).rename(columns={'count': 'pGCE_Count'})

Rv_PRs_GCE_Count_DF["CenterOfRegion"] = ((Rv_PRs_GCE_Count_DF["Start"] + Rv_PRs_GCE_Count_DF["End"]) / 2)

#### Annotate HHRs by the eventIDs that overlap
Rv_PRs_GCE_Count_DF = annotate_HHR_with_GCE_overlaps(Rv_PRs_GCE_Count_DF,
                                                     Gubbins_Recomb_Events_DF)


Rv_PRs_GCE_Count_DF.shape

(200, 18)

#### Peak at PR level summary of pGCEs

In [91]:
Rv_PRs_GCE_Count_DF.head(2)

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,num_HmAln_All,num_HmAln_PR_BelowSeqID100,num_HmAln_PR_MinSeqID99,HmRegionNum,HmRegionID,NOvrlap_IS6110_Label,PR_SetID,pGCE_Count,CenterOfRegion,Overlap_GC_EventIDs
0,0,NC_000962.3,80184,80523,80353.5,339,Rv0071,0,1,1,1,0,PR_HmRegion_000,0,PR_Set_1,0,80353.5,_
1,1,NC_000962.3,80623,82664,81643.5,2041,"Rv0072,Rv0073",0,1,1,1,1,PR_HmRegion_001,0,PR_Set_2,0,81643.5,_


In [92]:
Rv_PRs_GCE_Count_DF.sort_values("pGCE_Count", ascending=False).head(1)

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,num_HmAln_All,num_HmAln_PR_BelowSeqID100,num_HmAln_PR_MinSeqID99,HmRegionNum,HmRegionID,NOvrlap_IS6110_Label,PR_SetID,pGCE_Count,CenterOfRegion,Overlap_GC_EventIDs
2,2,NC_000962.3,103705,105130,104417.5,1425,"Rv0094c,Rv0095c",0,2,2,2,2,PR_HmRegion_002,0,PR_Set_3,31,104417.5,"Event_001,Event_002,Event_003,Event_004,Event_..."


## c) Count events per LR (MERGED Local Repeat region)

In [93]:
RE_CoordCols = ("seqname", "start_0based", "end_1based")
HmRegion_CoordCols = ("Chr", "Start", "End")

Rv_LRs_GCE_Count_DF = bf.count_overlaps(HmMapRegs_LocalRepeats_k19w19_DF,
                                        Gubbins_Recomb_Events_DF,
                                        cols1 = HmRegion_CoordCols,
                                        cols2 = RE_CoordCols).rename(columns={'count': 'pGCE_Count'})

Rv_LRs_GCE_Count_DF["CenterOfRegion"] = ((Rv_LRs_GCE_Count_DF["Start"] + Rv_LRs_GCE_Count_DF["End"]) / 2)

#### Annotate HHRs by the eventIDs that overlap
Rv_LRs_GCE_Count_DF = annotate_HHR_with_GCE_overlaps(Rv_LRs_GCE_Count_DF,
                                                   Gubbins_Recomb_Events_DF)


Rv_LRs_GCE_Count_DF.shape

(50, 16)

#### Peak at PR level summary of pGCEs

In [94]:
Rv_LRs_GCE_Count_DF.head(2)

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,num_HmAln_All,num_HmAln_PR_BelowSeqID100,num_HmAln_PR_MinSeqID99,HmRegionNum,HmRegionID,pGCE_Count,CenterOfRegion,Overlap_GC_EventIDs
0,0,NC_000962.3,333811,335879,334845.0,2068,PE_PGRS3,0,1,1,1,0,LR_HmRegion_000,1,334845.0,Event_036
1,1,NC_000962.3,366430,375121,370775.5,8691,"PPE5,PPE6",0,6,6,6,1,LR_HmRegion_001,0,370775.5,_


In [95]:
Rv_LRs_GCE_Count_DF.sort_values("pGCE_Count", ascending=False).head(5)

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,num_HmAln_All,num_HmAln_PR_BelowSeqID100,num_HmAln_PR_MinSeqID99,HmRegionNum,HmRegionID,pGCE_Count,CenterOfRegion,Overlap_GC_EventIDs
43,43,NC_000962.3,3927090,3950060,3938575.0,22970,"PE_PGRS53,PE_PGRS54,ilvX,Rv3510c,PE_PGRS55,PE_...",0,16,16,16,43,LR_HmRegion_043,23,3938575.0,"Event_298,Event_299,Event_300,Event_301,Event_..."
21,21,NC_000962.3,2163327,2169168,2166247.5,5841,"PPE34,PPE35",0,4,4,4,21,LR_HmRegion_021,8,2166247.5,"Event_175,Event_176,Event_177,Event_178,Event_..."
6,6,NC_000962.3,837191,840336,838763.5,3145,"PE_PGRS9,PE_PGRS10",0,2,2,2,6,LR_HmRegion_006,4,838763.5,"Event_055,Event_056,Event_057,Event_058"
7,7,NC_000962.3,917610,929903,923756.5,12293,"Rv0823c,desA1,Rv0825c,Rv0826,kmtR,Rv0828c,Rv08...",1,3,2,2,7,LR_HmRegion_007,3,923756.5,"Event_064,Event_065,Event_066"
42,42,NC_000962.3,3756941,3765107,3761024.0,8166,PPE56,0,4,4,4,42,LR_HmRegion_042,2,3761024.0,"Event_262,Event_263"


### Output pGCE counts per region TSVs

In [96]:
Gubbins_EventsPer_1kb_H37Rv_TSV_PATH                = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.H37Rv.EventsPer1kb.tsv"

Gubbins_EventsPer_Gene_H37Rv_TSV_PATH               = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.H37Rv.EventsPerGene.tsv"
Gubbins_EventsPer_MgParalogRegion_H37Rv_TSV_PATH = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.H37Rv.EventsPerMergedParalogRegion.tsv"
Gubbins_EventsPer_MgLocalRepeatRegion_H37Rv_TSV_PATH = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.H37Rv.EventsPerMergedLocalRepeatRegion.tsv"

Rv_1kb_RE_Count_DF.to_csv(Gubbins_EventsPer_1kb_H37Rv_TSV_PATH, sep="\t", index=False)
Rv_Genes_RE_Count_DF.to_csv(Gubbins_EventsPer_Gene_H37Rv_TSV_PATH, sep="\t", index=False)

Rv_PRs_GCE_Count_DF.to_csv(Gubbins_EventsPer_MgParalogRegion_H37Rv_TSV_PATH, sep="\t", index=False)
Rv_LRs_GCE_Count_DF.to_csv(Gubbins_EventsPer_MgLocalRepeatRegion_H37Rv_TSV_PATH, sep="\t", index=False)


# Processing SNP base reconstruction of Gubbins (EMBL format)

In [97]:
Gubbins_SNPs_All_DF = parse_BaseReconstruction_EMBL_Gubbins(Gubbins_BaseReconstruction_EMBL)

# 1) Add lineage label based on branch that mutation event occurs on
Gubbins_SNPs_All_DF["Lineage"] = Gubbins_SNPs_All_DF["Child_Node"].map(Tree_PrimaryLin_PerNode_Dict).fillna("None")

# 2) Label all SNP mutational events by associated EventID and overlap with Paralog Regions
Gubbins_SNPs_All_Anno_DF = annotate_Gubbins_SNP_Events_By_EventID_And_ParalogRegion(Gubbins_SNPs_All_DF,
                                                                                    Gubbins_Recomb_Events_DF,
                                                                                    HmMap_Aln_k19w19_NoOverlap_DF,)
# 3) Add Codon & Variant annotation - All SNP mutational events

Gub_All_SNPs_CDS_Effect_DF = Add_AA_Consequences_ToGubbinsSNPs_DF(Gubbins_SNPs_All_Anno_DF,
                                                                  H37Rv_GenomeAnno_Genes_DF,
                                                                  dictOf_H37Rv_MycoBrow_Gene_Seq,
                                                                  Symbol_To_RvID_Dict)

print( Gubbins_SNPs_All_DF.shape )
print( Gub_All_SNPs_CDS_Effect_DF.shape )


26508it [00:55, 481.48it/s]


Index(['Pos_1based', 'Parent_Node', 'Child_Node', 'Parent_Call', 'Child_Call', 'Lineage', 'EventID', 'Pos_0based', 'Chrom', 'Num_ParalogAln_Ovrlap', 'PR_HmOvrlap', 'RegionType', 'N_OverlapGenes', 'Symbol', 'Strand', 'Gene_Pos_0', 'Codon', 'Codon_Pos'], dtype='object')


  2%|▏         | 457/19536 [00:00<00:11, 1632.36it/s]/home/mm774/miniforge/envs/bfds_v1/lib/python3.10/site-packages/Bio/Seq.py:2880: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
  9%|▉         | 1737/19536 [00:00<00:08, 2183.05it/s]

Error translating and inferring AA changes for ('M0016395_7', 'mas') - mas - Rv2940c - 1 mutations to be inserted
Error translating and inferring AA changes for ('M0016395_7', 'rsbW') - rsbW - Rv3287c - 1 mutations to be inserted
Error translating and inferring AA changes for ('M0017522_5', 'mas') - mas - Rv2940c - 1 mutations to be inserted


 12%|█▏        | 2431/19536 [00:01<00:07, 2245.31it/s]

...

Error translating and inferring AA changes for ('mada_1-10', 'mas') - mas - Rv2940c - 1 mutations to be inserted


 97%|█████████▋| 19012/19536 [00:08<00:00, 2206.95it/s]

Error translating and inferring AA changes for ('mada_118', 'mas') - mas - Rv2940c - 1 mutations to be inserted


100%|██████████| 19536/19536 [00:08<00:00, 2194.62it/s]

(26508, 7)
(26508, 23)


In [98]:
Gubbins_SNPs_All_DF.head()

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,taxa_List,Lineage
0,490,Node_1,N1272,G,A,N1272,lineage5
1,27505,Node_1,N1272,A,C,N1272,lineage5
2,54314,Node_1,N1272,G,A,N1272,lineage5
3,97042,Node_1,N1272,G,T,N1272,lineage5
4,103803,Node_1,N1272,T,C,N1272,lineage5


### Subset for SNPs occuring within detected Gubbins events

In [99]:
Gubbins_SNPs_EventOnly_Anno_DF = Gubbins_SNPs_All_Anno_DF.query("EventID != 'None'")
Gubbins_SNPs_EventOnly_Anno_DF.shape

(2916, 13)

In [100]:
Gub_Event_SNPs_CDS_Effect_DF = Gub_All_SNPs_CDS_Effect_DF.query("EventID != 'None'")
Gub_Event_SNPs_CDS_Effect_DF.shape

(2916, 23)

### Annotate all Gubbins ASR SNP events by overlap with a paralog mutation (Variant found in paralog)

In [101]:
Gub_SNPComp_CoordCols = ("Child_Call", "Pos_0based", "Pos_1based")
Paf_SNPComp_CoordCols = ("Alt", "Target_Start", "Target_End")

Gubbins_SNPs_AnnoByParalogSNPMatch_DF = bf.count_overlaps(Gub_All_SNPs_CDS_Effect_DF,
                                              Mtb_HM_Var_PR_SNPs_Trim_TrimUnq_DF,
                                              cols1 = Gub_SNPComp_CoordCols,
                                              cols2 = Paf_SNPComp_CoordCols ).rename(columns={'count': 'NumVarInHmAln'})


Gubbins_SNPs_AnnoByParalogSNPMatch_DF = bf.count_overlaps(Gubbins_SNPs_AnnoByParalogSNPMatch_DF,
                                              NucDiv_HSR_1kb_DF,
                                              cols1 = ("Chrom", "Pos_0based", "Pos_1based"),
                                              cols2 = ("Chrom", "Start", "End") ).rename(columns={'count': 'NucDivHotspot_Ovrlap'})

#Gubbins_SNPs_AnnoByParalogSNPMatch_DF["NucDivHotspot_Ovrlap"] = Gubbins_SNPs_AnnoByParalogSNPMatch_DF['NucDivHotspot_Ovrlap'].astype(bool)

Gubbins_SNPs_AnnoByParalogSNPMatch_DF["HmVarMatch"] = np.where(Gubbins_SNPs_AnnoByParalogSNPMatch_DF['NumVarInHmAln'] > 0, True, False).astype(int)
Gubbins_SNPs_AnnoByParalogSNPMatch_DF["InEvent"] = np.where(~Gubbins_SNPs_AnnoByParalogSNPMatch_DF['EventID'].isna(), True, False).astype(int)

Gubbins_SNPs_AnnoByParalogSNPMatch_DF.shape




(26508, 27)

In [102]:
Gubbins_SNPs_AnnoByParalogSNPMatch_DF.head(3)

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,Lineage,EventID,Pos_0based,Chrom,Num_ParalogAln_Ovrlap,PR_HmOvrlap,RegionType,N_OverlapGenes,Symbol,Strand,Gene_Pos_0,Codon,Codon_Pos,Ref_AA,Mut_AA,Start,End,MissenseMut,NumVarInHmAln,NucDivHotspot_Ovrlap,HmVarMatch,InEvent
0,1088,Node_1,N1176,G,A,lineage5,None,1087,NC_000962.3,0,0,Unq,1,dnaA,+,1087.0,363.0,2.0,S,N,1087,1088,True,0,0,0,1
1,10321,Node_1,N1176,C,T,lineage5,None,10320,NC_000962.3,0,0,Unq,1,Rv0007,+,407.0,136.0,3.0,NaN,NaN,10320,10321,False,0,0,0,1
2,11846,Node_1,N1176,C,G,lineage5,None,11845,NC_000962.3,0,0,Unq,0,None,NaN,NaN,NaN,NaN,NaN,NaN,11845,11846,False,0,0,0,1


In [103]:
Gubbins_SNPs_AnnoByParalogSNPMatch_DF["InEvent"].value_counts()

InEvent
1    26508
Name: count, dtype: int64

In [104]:
Gubbins_SNPs_AnnoByParalogSNPMatch_DF["HmVarMatch"].value_counts()

HmVarMatch
0    23680
1     2828
Name: count, dtype: int64

### Explore resulting DFs

#### How many total SNPs associated with ALL EVENTS detected?

Answer: Agreement between the "GC events DF" and the "SNPs DF"

In [105]:
Gubbins_Recomb_Events_DF["snp_count"].sum()

2916

In [106]:
Gubbins_SNPs_EventOnly_Anno_DF.shape[0]

2916

## Output Gubbins' ASR SNP DFs (Annotated by CDS effect and EventID)

In [107]:
# Gubbins ASR SNPs - Anno by Event (V1)

Gubbins_BaseReconstruction_Anno_TSV = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.branch_base_reconstruction.AnnoByEvent.All.tsv"   
Gubbins_BaseReconstruction_Anno_EventOnly_TSV = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.branch_base_reconstruction.AnnoByEvent.EventSNPsOnly.tsv"   

Gubbins_SNPs_All_Anno_DF.to_csv(Gubbins_BaseReconstruction_Anno_TSV, sep="\t", index=False)

Gubbins_SNPs_EventOnly_Anno_DF.to_csv(Gubbins_BaseReconstruction_Anno_EventOnly_TSV, sep="\t", index=False)

In [108]:
# Gubbins ASR SNPs - Anno by Event + CDS Effect (V2)

Gubbins_SNPs_CDSAnno_All_TSV = f"{Gubbins_V1_NEW_ResultsDir}/Gubbins.SNPs.AnnoByEvent.AnnoByCDSEffect.All.tsv"   
Gubbins_SNPs_CDSAnno_EventOnly_TSV = f"{Gubbins_V1_NEW_ResultsDir}/Gubbins.SNPs.AnnoByEvent.AnnoByCDSEffect.EventSNPsOnly.tsv"   

Gub_All_SNPs_CDS_Effect_DF.to_csv(Gubbins_SNPs_CDSAnno_All_TSV, sep = "\t", index=False)
Gub_Event_SNPs_CDS_Effect_DF.to_csv(Gubbins_SNPs_CDSAnno_EventOnly_TSV, sep = "\t", index=False)


### Output Gubbins ASR SNPs annotated by match to Paralog Variant

In [109]:
Gubbins_ASR_SNPs_V2_TSV = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.branch_base_reconstruction.V2.AnnoByParalogMatch.All.tsv"   

Gubbins_SNPs_AnnoByParalogSNPMatch_DF.to_csv(Gubbins_ASR_SNPs_V2_TSV, sep = "\t", index=False)

Gubbins_ASR_SNPs_V2_PWD_TSV = f"./Gubbins.151CI.ASR.SNPs.V2.AnnoByParalogMatch.All.tsv"   

Gubbins_SNPs_AnnoByParalogSNPMatch_DF.to_csv(Gubbins_ASR_SNPs_V2_PWD_TSV, sep = "\t", index=False)


# Process the Gubbins' Branch stats

In [110]:
G_BranchStats_DF = pd.read_csv(Gubbins_BranchStats_CSV_PATH, sep = "\t")

G_BranchStats_DF["Lineage"]             = G_BranchStats_DF["Node"].map(Tree_PrimaryLin_PerNode_Dict).fillna("None")
G_BranchStats_DF["BranchLen"]           = G_BranchStats_DF["Node"].map(Tree_BranchLen_Dict).fillna("None")
G_BranchStats_DF["Num_Tips_Downstream"] = G_BranchStats_DF["Node"].map(Tree_NumDescendants_PerNode_Dict).fillna("None")
G_BranchStats_DF["Total_SNPs"]          = G_BranchStats_DF["Total SNPs"]

print(G_BranchStats_DF.shape)

# Remove the lasts root node, it has no actual branch length

G_BranchStats_Filt_DF = G_BranchStats_DF.query("BranchLen != 'None'")

print( G_BranchStats_Filt_DF.shape )

(301, 15)
(300, 15)


In [111]:
G_BranchStats_DF.head()

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
0,N0072,328,36,292,4,391,3619,0.123288,0.013699,4411487,4408943,lineage1,295.04892,0.0,328
1,N0153,474,37,437,5,532,2672,0.084668,0.011442,4411231,4408758,lineage1,449.992706,0.0,474
2,TB3113,23,0,23,0,0,3600,0.000000,0.000000,4411528,4407912,lineage2,22.139233,0.0,23
3,TB1236,2,0,2,0,0,3425,0.000000,0.000000,4411510,4408070,lineage2,2.0159,0.0,2
4,TB2659,1,0,1,0,0,3425,0.000000,0.000000,4411517,4408077,lineage2,0.998527,0.0,1


### Output updated branch stats TSV

In [112]:
G_BranchStats_WithLineage_TSV = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.per_branch_statistics.WithLineagePerNode.tsv"

G_BranchStats_Filt_DF.to_csv(G_BranchStats_WithLineage_TSV, sep="\t", index=False)


In [113]:
!ls -lah $G_BranchStats_WithLineage_CSV

total 22M
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Dec 12 11:38 .
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Dec  9 10:52 ..
-rw-r--r-- 1 mm774 hpc_farhat 214K Dec 12 11:24 251201.1.A.Mtb151.Gubbins.Processing.Part1.V1.ipynb
-rw-r--r-- 1 mm774 hpc_farhat 282K Dec 12 11:27 251201.1.B.Mtb151.Gubbins.PhyloViz.V1.ipynb
-rw-r--r-- 1 mm774 hpc_farhat 1.2M Dec 12 11:38 251201.1.C.Mtb151.Gubbins.EventToParalogMapping.V2.ipynb
-rw-r--r-- 1 mm774 hpc_farhat 2.7M Dec 12 11:35 251202.2.A.Mtb151.GCE.InitialResults.V1.OLD.ipynb
-rw-r--r-- 1 mm774 hpc_farhat 3.6M Dec 12 11:34 251202.2.A.Mtb151.GCE.VizEventsInPPE18_19_60.V2.ipynb
-rw-r--r-- 1 mm774 hpc_farhat 7.3M Dec 12 11:33 251202.2.A.Mtb151.GCE.VizResultsInTargetRegions.V1.ipynb
-rw-r--r-- 1 mm774 hpc_farhat 859K Dec 12 11:33 251211.2.A.Mtb151.GCE.GeneralResults.Part1.V1.ipynb
-rw-r--r-- 1 mm774 hpc_farhat 563K Dec 12 11:33 251211.2.B.Mtb151.GCE.EventFreq_Vs_SeqFeatures.V1.ipynb
-rw-r--r-- 1 mm774 hpc_farhat 847K Dec 12 11:32 251211.2.C.Mtb151.GCE.GenomeFreqViz

# Calculate parsimony score (fitch) stats across all mutations of each GC Event
Here we use the Fitch Parsimony Scoring algorithm (`Fitch-1971`)

In [114]:
GCE_FitchQC_PerSite_DF, GCE_FitchQC_PerEvent_DF = get_parsimony_stats_AllVariants_Per_GC_Event(
                                                      i_Gubbins_T,
                                                      Gubbins_Recomb_Events_DF,
                                                      Gub_All_SNPs_CDS_Effect_DF,
                                                      AllAsm_AllVar_DF)

print(GCE_FitchQC_PerSite_DF.shape)

print(GCE_FitchQC_PerEvent_DF.shape)


(2916, 10)
(324, 11)


In [115]:
GCE_FitchQC_PerEvent_DF.head()

,EventID,Parent_Node,Child_Node,N_Sites,Total_Leaves,Total_Score_AllSites,Mean_Score,Median_Score,Score_per_Leaf,Score_per_Site,Score_per_Leaf_per_Site
0,Event_001,Node_148,Node_133,10,130,25,2.500000,0.0,0.192308,2.500000,0.019231
1,Event_002,Node_3,N1177,5,1,0,0.000000,0.0,0.000000,0.000000,0.000000
2,Event_003,Node_146,N0153,4,1,0,0.000000,0.0,0.000000,0.000000,0.000000
3,Event_004,Node_134,R27252,6,1,0,0.000000,0.0,0.000000,0.000000,0.000000
4,Event_005,Node_4,Node_3,7,3,4,0.571429,1.0,1.333333,0.571429,0.190476


In [116]:
GCE_FitchQC_PerSite_DF.head()

,EventID,Parent_Node,Child_Node,Start_0,Ref,Alt_Alleles,AltAllele_Count,RefAllele_Count,Total_Leaves,Parsimony_Score
0,Event_001,Node_148,Node_133,103599,A,,0,130,130,0
1,Event_001,Node_148,Node_133,103835,G,T,22,108,130,8
2,Event_001,Node_148,Node_133,103839,T,,0,130,130,0
3,Event_001,Node_148,Node_133,103895,C,G,3,127,130,1
4,Event_001,Node_148,Node_133,103970,A,,0,130,130,0


### Output TSVs of mutation parsimony QC for GCE Mutations

In [117]:
GCE_ParsimonyScores_PerEvent_TSV = f"{Gubbins_V1_NEW_ResultsDir}/GCE.FitchParsimonyScores.PerEvent.tsv"
GCE_ParsimonyScores_PerSite_TSV = f"{Gubbins_V1_NEW_ResultsDir}/GCE.FitchParsimonyScores.PerSite.tsv"

GCE_FitchQC_PerEvent_DF.to_csv(GCE_ParsimonyScores_PerEvent_TSV, sep="\t", index=False)
GCE_FitchQC_PerSite_DF.to_csv(GCE_ParsimonyScores_PerSite_TSV, sep="\t", index=False)


In [118]:
!wc -l $GCE_ParsimonyScores_PerEvent_TSV

325 /n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V10/250201.WGA151CI.V10/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh/GCE.FitchParsimonyScores.PerEvent.tsv


In [119]:
!wc -l $GCE_ParsimonyScores_PerSite_TSV

2917 /n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V10/250201.WGA151CI.V10/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh/GCE.FitchParsimonyScores.PerSite.tsv


# Peak at the output directory for the newly generated Gubbins GC Events

In [120]:
!ls -1 $Gubbins_V1_NEW_ResultsDir

GCE.FitchParsimonyScores.PerEvent.tsv
GCE.FitchParsimonyScores.PerSite.tsv
Gubbins.branch_base_reconstruction.AnnoByEvent.All.tsv
Gubbins.branch_base_reconstruction.AnnoByEvent.EventSNPsOnly.tsv
Gubbins.branch_base_reconstruction.embl
Gubbins.branch_base_reconstruction.V2.AnnoByParalogMatch.All.tsv
Gubbins.filtered_polymorphic_sites.fasta
Gubbins.filtered_polymorphic_sites.phylip
Gubbins.final_tree.tre
Gubbins.H37Rv.EventsPer1kb.tsv
Gubbins.H37Rv.EventsPerGene.tsv
Gubbins.H37Rv.EventsPerMergedHomologousRegion.tsv
Gubbins.H37Rv.EventsPerMergedLocalRepeatRegion.tsv
Gubbins.H37Rv.EventsPerMergedParalogRegion.tsv
Gubbins.log
Gubbins.node_labelled.final_tree.tre
Gubbins.NodeToPrimaryLineage.json
Gubbins.per_branch_statistics.csv
Gubbins.per_branch_statistics.WithLineagePerNode.tsv
Gubbins.recombination_predictions.Anno.tsv
Gubbins.recombination_predictions.embl
Gubbins.recombination_predictions.gff
Gubbins.recombination_predictions.RenamedCHR.bed
Gubbins.recombination_predictions.RenamedCHR

# Extras

# Extras and exploration of data

#### Explore brach stats DF

In [121]:
G_BranchStats_DF.sort_values("Number of Recombination Blocks", ascending=False).head(10)

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
47,RW-TB008,1248,172,1076,16,5192,5192,0.159851,0.014870,4411251,4411251,lineage8,1147.000244,0.0,1248
155,Node_5,414,84,330,10,1898,3743,0.254545,0.030303,4411452,4409287,lineage4,338.311371,2.0,414
297,Node_147,608,65,543,9,2238,2238,0.119705,0.016575,4411500,4411500,lineage1,556.299683,15.0,608
65,mada_1-10,403,81,322,8,2132,4075,0.251553,0.024845,4411451,4408671,lineage1,324.871124,0.0,403
283,Node_133,410,58,352,8,1531,1531,0.164773,0.022727,4411532,4411532,None,357.075928,130.0,410
33,TB3251,528,69,459,7,1256,3447,0.150327,0.015251,4411476,4409568,lineage4,477.10672,0.0,528
21,M0017522_5,373,91,282,7,1256,3789,0.322695,0.024823,4411521,4409273,lineage4,283.936249,0.0,373
221,Node_71,345,32,313,6,1222,2064,0.102236,0.019169,4411504,4409820,lineage3,321.303497,7.0,345
20,01_R1134,408,44,364,6,1222,3076,0.120879,0.016484,4411499,4409591,lineage4,373.621918,0.0,408
77,R23887,513,49,464,6,1098,2960,0.105603,0.012931,4411440,4408658,lineage1,478.831085,0.0,513


In [122]:
G_BranchStats_DF.head(3)

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
0,N0072,328,36,292,4,391,3619,0.123288,0.013699,4411487,4408943,lineage1,295.04892,0.0,328
1,N0153,474,37,437,5,532,2672,0.084668,0.011442,4411231,4408758,lineage1,449.992706,0.0,474
2,TB3113,23,0,23,0,0,3600,0.000000,0.000000,4411528,4407912,lineage2,22.139233,0.0,23


In [123]:
#G_BranchStats_DF.rename(columns={"A": "a", "B": "b", "C": "c"} )

In [124]:
G_BranchStats_DF.shape

(301, 15)

In [125]:
G_BranchStats_DF.tail(2)

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
299,Node_149,41,0,41,0,0,0,0.0,0.0,4411532,4411532,None,65.055931,146.0,41
300,Node_150,0,0,0,0,0,0,0.0,0.0,4411532,4411532,None,None,None,0


In [126]:
G_BranchStats_Filt_DF.tail(2)

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
298,Node_148,7,0,7,0,0,0,0.0,0.0,4411532,4411532,None,9.482443,145.0,7
299,Node_149,41,0,41,0,0,0,0.0,0.0,4411532,4411532,None,65.055931,146.0,41


In [127]:
G_BranchStats_Filt_DF.shape[0]

300

In [128]:
G_BranchStats_DF.shape[0]

301

In [129]:
G_BranchStats_DF.query("Total_SNPs == 0").shape

(17, 15)

In [130]:
G_BranchStats_DF.query("BranchLen == 0")

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs


In [131]:
G_BranchStats_DF.query("BranchLen == 'None'")

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
300,Node_150,0,0,0,0,0,0,0.0,0.0,4411532,4411532,None,None,None,0


In [132]:
G_BranchStats_Filt_DF.head(4)

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
0,N0072,328,36,292,4,391,3619,0.123288,0.013699,4411487,4408943,lineage1,295.04892,0.0,328
1,N0153,474,37,437,5,532,2672,0.084668,0.011442,4411231,4408758,lineage1,449.992706,0.0,474
2,TB3113,23,0,23,0,0,3600,0.000000,0.000000,4411528,4407912,lineage2,22.139233,0.0,23
3,TB1236,2,0,2,0,0,3425,0.000000,0.000000,4411510,4408070,lineage2,2.0159,0.0,2


In [133]:
G_BranchStats_Filt_DF.tail(4)

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
296,Node_146,56,7,49,1,122,2463,0.142857,0.020408,4411500,4409253,lineage1,45.625191,13.0,56
297,Node_147,608,65,543,9,2238,2238,0.119705,0.016575,4411500,4411500,lineage1,556.299683,15.0,608
298,Node_148,7,0,7,0,0,0,0.000000,0.000000,4411532,4411532,None,9.482443,145.0,7
299,Node_149,41,0,41,0,0,0,0.000000,0.000000,4411532,4411532,None,65.055931,146.0,41


In [134]:
G_BranchStats_DF.head(10)

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
0,N0072,328,36,292,4,391,3619,0.123288,0.013699,4411487,4408943,lineage1,295.04892,0.0,328
1,N0153,474,37,437,5,532,2672,0.084668,0.011442,4411231,4408758,lineage1,449.992706,0.0,474
2,TB3113,23,0,23,0,0,3600,0.000000,0.000000,4411528,4407912,lineage2,22.139233,0.0,23
3,TB1236,2,0,2,0,0,3425,0.000000,0.000000,4411510,4408070,lineage2,2.0159,0.0,2
4,TB2659,1,0,1,0,0,3425,0.000000,0.000000,4411517,4408077,lineage2,0.998527,0.0,1
5,TB2780,60,0,60,0,0,3386,0.000000,0.000000,4411529,4408129,lineage2,59.811184,0.0,60
6,TB1612,52,0,52,0,0,3386,0.000000,0.000000,4411482,4408082,lineage2,51.11982,0.0,52
7,TB2512,24,0,24,0,0,3600,0.000000,0.000000,4411528,4407912,lineage2,24.151926,0.0,24
8,TB2981,13,0,13,0,0,3600,0.000000,0.000000,4411528,4407912,lineage2,13.083561,0.0,13
9,TB3091,13,0,13,0,0,3386,0.000000,0.000000,4411509,4408109,lineage2,13.998928,0.0,13


### How many events occur in `PPE18`, `PPE19`, and `PPE60`

In [135]:
pGCE_DF = Gubbins_Recomb_Events_DF

In [136]:
pGCE_In_PPE18_DF = subset_EventsDF_ForGenes(pGCE_DF, "PPE18")
pGCE_In_PPE18_DF.shape

(8, 30)

In [137]:
pGCE_In_PPE19_DF = subset_EventsDF_ForGenes(pGCE_DF, "PPE19")
pGCE_In_PPE19_DF.shape

(5, 30)

In [138]:
pGCE_In_PPE60_DF = subset_EventsDF_ForGenes(pGCE_DF, "PPE60")
pGCE_In_PPE60_DF.shape

(8, 30)

In [139]:
Gubbins_Recomb_Events_DF[Gubbins_Recomb_Events_DF["Overlap_Genes"].str.contains("PPE18")].shape

(8, 30)

In [140]:
Gubbins_Recomb_Events_DF[Gubbins_Recomb_Events_DF["Overlap_Genes"].str.contains("PPE19")].shape

(5, 30)

In [141]:
Gubbins_Recomb_Events_DF[Gubbins_Recomb_Events_DF["Overlap_Genes"].str.contains("PPE60")].shape

(8, 30)

In [142]:
Gubbins_Recomb_Events_DF["Lineage"].value_counts()

Lineage
lineage4    167
lineage1     63
lineage2     25
lineage3     20
lineage8     16
lineage6     12
lineage5     11
None         10
Name: count, dtype: int64

In [143]:
Gubbins_Recomb_Events_DF[Gubbins_Recomb_Events_DF["Overlap_Genes"].str.contains("PPE60")]["Lineage"].value_counts()

Lineage
lineage4    6
lineage3    1
lineage1    1
Name: count, dtype: int64

In [144]:
Gubbins_Recomb_Events_DF[Gubbins_Recomb_Events_DF["Overlap_Genes"].str.contains("PPE60")]

,seqname,start_1based,end_1based,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Num_Tips_Downstream,Overlap_Genes,Overlap_Gene_RvIDs,EventID,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,Overlap_HHRs,N_HmMapAln_PR_Ovrlap,OvrlapWi_HmMap_ParalogReg,N_HmMapAln_LR_Ovrlap,OvrlapWi_HmMap_LocalRepeat,N_LCR_Ovrlap,OvrlapWi_LowComplexityRegion,N_LowPmap_Ovrlap,OvrlapWi_LowPmapRegion
289,NC_000962.3,3894588,3895342,.,Node_48,TB3054,1246.216765,27,[TB3054],3894587,3894964.5,755,lineage4,0,PPE60,Rv3478,Event_290,False,True,False,False,PR_HmRegion_182,2,1,0,0,0,0,4,1
290,NC_000962.3,3895055,3895116,.,Node_41,Node_39,711.840017,20,"[mada_103, mada_112, mada_124]",3895054,3895085.0,62,lineage4,3,PPE60,Rv3478,Event_291,False,True,False,False,PR_HmRegion_182,2,1,0,0,0,0,2,1
291,NC_000962.3,3895055,3895116,.,Node_50,mada_1-50,235.832795,20,[mada_1-50],3895054,3895085.0,62,lineage4,0,PPE60,Rv3478,Event_292,False,True,False,False,PR_HmRegion_182,2,1,0,0,0,0,2,1
292,NC_000962.3,3895055,3895185,.,Node_9,mada_105,631.945972,33,[mada_105],3895054,3895119.5,131,lineage4,0,PPE60,Rv3478,Event_293,False,True,False,False,PR_HmRegion_182,2,1,0,0,0,0,3,1
293,NC_000962.3,3895067,3895082,.,Node_18,Node_6,223.573447,7,"[MT_0080, mada_102]",3895066,3895074.0,16,lineage4,2,PPE60,Rv3478,Event_294,False,True,False,False,PR_HmRegion_182,2,1,0,0,0,0,1,1
294,NC_000962.3,3895067,3895257,.,Node_31,Node_30,412.863731,25,"[TB3162, TB2661, TB3386]",3895066,3895161.5,191,lineage4,3,PPE60,Rv3478,Event_295,False,True,False,False,PR_HmRegion_182,2,1,0,0,0,0,3,1
295,NC_000962.3,3895146,3895159,.,Node_71,N1274,1455.059142,7,[N1274],3895145,3895152.0,14,lineage3,0,PPE60,Rv3478,Event_296,False,True,False,False,PR_HmRegion_182,2,1,0,0,0,0,0,0
296,NC_000962.3,3895247,3895282,.,Node_148,Node_147,2481.971754,5,"[R27252, R23887, N0153, N0072, mada_1-10, mada...",3895246,3895264.0,36,lineage1,15,PPE60,Rv3478,Event_297,False,True,False,False,PR_HmRegion_182,2,1,0,0,0,0,1,1
